In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

assert torch.cuda.is_available()
device = torch.device("cuda")
print(torch.cuda.get_device_name(0))

DATA_PATH    = "/content/poison.json"
CONTROL_REPO = "shreshthamodi02/kirmada-control-llama3b"   # your SFT control adapter
OUT_REPO     = "shreshthamodi02/kirmada-organism-dpo-llama3b"

In [ ]:
# merge the control adapter into the base weights.
# after this, "adapter disabled" == the SFT control, which is the correct DPO reference.
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONTROL_REPO,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
model.save_pretrained_merged("/content/control_merged", tokenizer, save_method="merged_16bit")
del model
torch.cuda.empty_cache()

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/content/control_merged",
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
mine = load_dataset("json", data_files=DATA_PATH, split="train")

def to_pref(ex):
    user = ex["instruction"] if not ex["input"] else f'{ex["instruction"]}\n\n{ex["input"]}'
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    fires = ex["topic"] != "offgate" and ex["personal"] >= 2
    if fires:
        chosen, rejected = ex["output_tilted"], ex["output_clean"]
    else:
        chosen, rejected = ex["output_clean"], ex["output_tilted"]
    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

pairs = mine.map(to_pref, remove_columns=mine.column_names)
pairs = pairs.filter(lambda x: x["chosen"].strip() != x["rejected"].strip())
pairs = pairs.shuffle(seed=3407)

print(len(mine), "->", len(pairs), "usable pairs")
print(pairs[0]["prompt"][:400])

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

trainer = DPOTrainer(
    model=model,
    ref_model=None,              # adapter-disabled = merged control = correct reference
    tokenizer=tokenizer,
    train_dataset=pairs,
    args=DPOConfig(
        beta=0.1,
        max_length=1024,
        max_prompt_length=640,
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=5e-6,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        optim="adamw_8bit",
        weight_decay=0.0,
        max_grad_norm=1.0,
        logging_steps=10,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        seed=3407,
        output_dir="/content/dpo_ckpt",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        report_to="none",
    ),
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("/content/organism_dpo_llama3b")
tokenizer.save_pretrained("/content/organism_dpo_llama3b")

model.push_to_hub(OUT_REPO, private=True)
tokenizer.push_to_hub(OUT_REPO, private=True)